In [ ]:
# # Kernel olarak nndet_venv seçin
# # veya başa ekleyin:
# import sys
# sys.path.insert(0, '/Users/ramazanpolat/Desktop/datasets/abdomen/nnDetection')
# sys.path.insert(0, '/Users/ramazanpolat/Desktop/datasets/abdomen/nndet_venv/lib/python3.9/site-packages')


# nnDetection 3D Eğitimi (Fold 0)

**Amaç:** Faz 3'teki 2D YOLO baseline'a karşılık 3D volumetric nesne saptama uygulamak. nnDetection (Baumgartner 2021), nnU-Net protokolünü medikal nesne saptamaya taşıyan self-configuring çerçevedir.

**Adımlar**
1. Ortam + nnDetection kontrolü
2. DICOM seri → NIfTI 3D hacim dönüşümü
3. 2D BBox → 3D kutu yükseltme (ardışık-kesit gruplamayla)
4. Task100_Abdomen veri formatı + dataset.json
5. nnDetection plan + preprocess
6. Fold 0 eğitim
7. Val 3D çıkarım + 2D-projeksiyon F1@IoU

**Kaynak uyarısı:** Kaggle T4 (15GB) için patch_size=96³ önerilir; A100 (40GB) ile 128³ veya 160³ mümkün.

## 1. Ortam ve nnDetection Kontrolü

In [1]:
import sys

!{sys.executable} -m pip install pydicom
!{sys.executable} -m pip install monai
!{sys.executable} -m pip install SimpleITK

print("Gerekli kütüphaneler başarıyla yüklendi.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 13.2 MB/s eta 0:00:00
Gerekli kütüphaneler başarıyla yüklendi.


In [2]:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [3]:
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple

import sys, shutil, subprocess


In [4]:
SUPER_CLASSES: List[str] = [
    "acute_cholecystitis",        # 0
    "kidney_ureter_stone",        # 1
    "acute_pancreatitis",         # 2
    "aortic_aneurysm_dissection", # 3
    "acute_appendicitis",         # 4
    "acute_diverticulitis",       # 5
]

In [5]:


ROOT = Path(r"/content")
DATA_ROOT = Path(r"/content/drive/MyDrive/Abdomen")
NIFTI_DIR = Path(r"/content/drive/MyDrive/Abdomen/nifti")

EGITIM_DIR = Path(DATA_ROOT / "Egitim Verisi")
YARISMA_DIR = Path(DATA_ROOT / "Test Verisi")



OUT_DIR      = Path(str(ROOT / "outputs"))
SPLIT_DIR    = Path(str(OUT_DIR / "splits"))


NND_ROOT = OUT_DIR / "nndet"
os.environ["det_data"]    = str(NND_ROOT / "raw")
os.environ["det_models"]  = str(NND_ROOT / "models")
os.environ["OMP_NUM_THREADS"] = "1"
for p in [NND_ROOT, Path(os.environ["det_data"]), Path(os.environ["det_models"])]:
    p.mkdir(parents=True, exist_ok=True)
print("det_data  :", os.environ["det_data"])
print("det_models:", os.environ["det_models"])

det_data  : /content/outputs/nndet/raw
det_models: /content/outputs/nndet/models


In [6]:
try:
    import nndet
    print("nnDetection yüklü:", getattr(nndet, "__version__", "ok"))
except ImportError:
    print("⚠️  nndet yüklü değil. Kurulum:")
    print("   pip install nndet")
    print("   # veya: git clone https://github.com/MIC-DKFZ/nnDetection.git && pip install -e nnDetection")
    print("Windows: Docker veya WSL2 önerilir.")

⚠️  nndet yüklü değil. Kurulum:
   pip install nndet
   # veya: git clone https://github.com/MIC-DKFZ/nnDetection.git && pip install -e nnDetection
Windows: Docker veya WSL2 önerilir.


### nnDetection Kurulumu

nnDetection kütüphanesini GitHub'dan klonlayıp kurulumunu yapacağız.

In [7]:
from pathlib import Path
import re

NNDET_REPO_DIR = Path("/content/nnDetection")

if not NNDET_REPO_DIR.exists():
    print(f"Cloning nnDetection into {NNDET_REPO_DIR}...")
    !git clone https://github.com/MIC-DKFZ/nnDetection.git {NNDET_REPO_DIR}
else:
    print(f"nnDetection repository already exists at {NNDET_REPO_DIR}.")

%cd {NNDET_REPO_DIR}

# Modify setup.py to remove SimpleITK dependency
setup_py_path = NNDET_REPO_DIR / "setup.py"
modified = False
if setup_py_path.exists():
    original_content = setup_py_path.read_text()
    new_lines = []

    in_install_requires_block = False
    for line in original_content.splitlines():
        if "install_requires=" in line:
            in_install_requires_block = True
            # Check for single-line definition of install_requires
            if "]" in line and "[" in line and line.index("[") < line.index("]") and line.count("[") == 1 and line.count("]") == 1:
                # If it's a single line, process it immediately and reset flag
                list_str_start = line.find("[")
                list_str_end = line.find("]", list_str_start)
                list_content = line[list_str_start + 1 : list_str_end].strip()
                elements = [el.strip() for el in list_content.split(',') if el.strip()]
                new_elements = []
                removed_current_line = False
                for el in elements:
                    if re.search(r"['\"]SimpleITK(?:[<>=~!]+.*?)?['\"]", el):
                        print(f"Removing SimpleITK dependency from single-line install_requires: {el.strip()}")
                        modified = True
                    else:
                        new_elements.append(el)
                new_list_str = "[" + ", ".join(new_elements) + "]"
                new_line_content = line[:list_str_start] + new_list_str + line[list_str_end + 1:]
                new_lines.append(new_line_content)
                in_install_requires_block = False # Exit block as it was a single line
            else:
                # Multi-line install_requires starts here
                new_lines.append(line)
        elif in_install_requires_block:
            if "]" in line and "install_requires=" not in line: # End of multi-line list
                if re.search(r"['\"]SimpleITK(?:[<>=~!]+.*?)?['\"]", line):
                    print(f"Removing SimpleITK dependency from multi-line install_requires: {line.strip()}")
                    modified = True
                else:
                    new_lines.append(line)
                in_install_requires_block = False
            elif re.search(r"['\"]SimpleITK(?:[<>=~!]+.*?)?['\"]", line):
                print(f"Removing SimpleITK dependency from multi-line install_requires: {line.strip()}")
                modified = True
            else:
                new_lines.append(line)
        else:
            new_lines.append(line)

    if modified:
        new_content = "\n".join(new_lines)
        # Clean up potential double commas and leading/trailing commas after removal
        new_content = re.sub(r",\s*,", ",", new_content)
        new_content = re.sub(r"install_requires=\[\s*,\s*", "install_requires=[", new_content)
        new_content = re.sub(r",\s*\]", "]", new_content)
        setup_py_path.write_text(new_content)
        print("setup.py modified successfully.")
    else:
        print("SimpleITK dependency not found in install_requires list.")
else:
    print("setup.py not found at expected path.")

# Now install nnDetection
!pip install -e .

print("nnDetection kurulumu tamamlandı.")

Cloning nnDetection into /content/nnDetection...
Cloning into '/content/nnDetection'...
remote: Enumerating objects: 1535, done.
remote: Counting objects: 100% (359/359), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 1535 (delta 291), reused 247 (delta 245), pack-reused 1176 (from 2)
Receiving objects: 100% (1535/1535), 1.34 MiB | 9.07 MiB/s, done.
Resolving deltas: 100% (894/894), done.
/content/nnDetection
SimpleITK dependency not found in install_requires list.
Obtaining file:///content/nnDetection
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.6/276.6 kB 8.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit

In [8]:
import nndet
print("nnDetection yüklü: ", getattr(nndet, "__version__", "ok"))

%cd /content

nnDetection yüklü:  ok
/content


## 2. DICOM Seri → NIfTI Hacim

In [ ]:
# !pip install SimpleITK --only-binary=:all:

In [9]:
import SimpleITK as sitk


In [10]:
def load_fold(fold: int, split: str) -> List[int]:
    """fold ∈ [0, n_splits), split ∈ {'train','val'}."""
    p = SPLIT_DIR / f"fold{fold}_{split}.csv"
    return pd.read_csv(p)["Case Number"].astype(int).tolist()

## 3. 2D BBox → 3D Kutu Yükseltme

`Bilgi.xlsx`'teki tüm annotasyonlar 2D. 3D detection için ardışık kesit BBox'larını birleştirip 3D kutu üretiyoruz:
- Aynı vaka + aynı sınıf
- Ardışık veya yakın kesit (Δz ≤ 2)
- 2D IoU ≥ 0.3 → aynı lezyon

In [ ]:
import numpy as np
import pydicom

def _2d_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw * ih
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    return inter / max(ua + ub - inter, 1e-6)

def case_image_id_to_z(case):
    cd = EGITIM_DIR / str(case)
    files = sorted(
        [p for p in cd.glob("*.dcm") if not p.stem.startswith(".")],
        key=lambda p: int(p.stem)
    )
    positions = []
    for p in files:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        zpos = float(getattr(ds, 'ImagePositionPatient', [0,0,int(p.stem)])[2])
        positions.append((int(p.stem), zpos))
    positions.sort(key=lambda x: x[1])
    return {img_id: idx for idx, (img_id, _) in enumerate(positions)}

def lift_2d_to_3d(manifest, case, delta_z=2, iou_th=0.3):
    z_map = case_image_id_to_z(case)
    sub = manifest[(manifest['case'] == case) &
                   (manifest['bboxes'].fillna('').astype(str) != '')]
    items = []
    for _, row in sub.iterrows():
        z = z_map.get(int(row['image_id']))
        if z is None: continue
        for bb_str in str(row['bboxes']).split('|'):
            parts = bb_str.split(',')
            if len(parts) < 5: continue
            sid = int(parts[0]); x1,y1,x2,y2 = map(int, parts[1:5])
            items.append((sid, x1, y1, x2, y2, z))
    boxes_3d = []
    for sid in set(it[0] for it in items):
        cls_items = sorted([it for it in items if it[0] == sid], key=lambda x: x[5])
        used = set()
        for i, it in enumerate(cls_items):
            if i in used: continue
            grp = [it]; used.add(i)
            for j in range(i+1, len(cls_items)):
                if j in used: continue
                last = grp[-1]
                if cls_items[j][5] - last[5] <= delta_z:
                    if _2d_iou(last[1:5], cls_items[j][1:5]) >= iou_th:
                        grp.append(cls_items[j]); used.add(j)
                else: break
            xs1 = min(g[1] for g in grp); ys1 = min(g[2] for g in grp)
            xs2 = max(g[3] for g in grp); ys2 = max(g[4] for g in grp)
            zs1 = min(g[5] for g in grp); zs2 = max(g[5] for g in grp)
            boxes_3d.append({"class": sid, "x1": xs1, "y1": ys1, "z1": zs1,
                             "x2": xs2, "y2": ys2, "z2": zs2,
                             "n_slices": len(grp)})
    return boxes_3d

manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
demo_case = fold_cases[0]
demo_boxes = lift_2d_to_3d(manifest, demo_case)
print(f"Vaka {demo_case}: {len(demo_boxes)} adet 3D kutu")
for b in demo_boxes[:5]:
    print(f"  sınıf {b['class']} ({SUPER_CLASSES[b['class']]}): "
          f"({b['x1']},{b['y1']},{b['z1']}) → ({b['x2']},{b['y2']},{b['z2']}), {b['n_slices']} kesit")

## 4. nnDetection Veri Formatı (Task100_Abdomen)

In [ ]:
import json
import numpy as np
import SimpleITK as sitk

TASK_NAME = "Task100_Abdomen"
TASK_DIR = Path(os.environ["det_data"]) / TASK_NAME / "raw_splitted"
(TASK_DIR / "imagesTr").mkdir(parents=True, exist_ok=True)
(TASK_DIR / "imagesTs").mkdir(parents=True, exist_ok=True)
(TASK_DIR / "labelsTr").mkdir(parents=True, exist_ok=True)

train_cases = load_fold(fold, "train")
val_cases   = load_fold(fold, "val")

def link_or_copy(src, dst):
    if dst.exists(): return
    try: os.symlink(src, dst)
    except (OSError, NotImplementedError): shutil.copy2(src, dst)

def prep_case(case, split):
    src_nii = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
    if not src_nii.exists(): return f"missing:{case}"

    target_img = TASK_DIR / ("imagesTr" if split != "test" else "imagesTs") / f"case_{case:05d}_0000.nii.gz"
    link_or_copy(src_nii, target_img)
    if split == "test": return "ok"

    label_nii_path  = TASK_DIR / "labelsTr" / f"case_{case:05d}.nii.gz"
    label_json_path = TASK_DIR / "labelsTr" / f"case_{case:05d}.json"
    if label_nii_path.exists() and label_json_path.exists():
        return "skip"

    boxes_3d = lift_2d_to_3d(manifest, case)

    img_itk  = sitk.ReadImage(str(src_nii))
    img_arr  = sitk.GetArrayFromImage(img_itk)   # [z, y, x]
    mask_arr = np.zeros(img_arr.shape, dtype=np.uint8)

    instances_json = {}
    coords = []
    for i, b in enumerate(boxes_3d):
        inst_id = i + 1
        z1 = int(b["z1"]); z2 = min(int(b["z2"]) + 1, mask_arr.shape[0])
        y1 = int(b["y1"]); y2 = min(int(b["y2"]) + 1, mask_arr.shape[1])
        x1 = int(b["x1"]); x2 = min(int(b["x2"]) + 1, mask_arr.shape[2])
        mask_arr[z1:z2, y1:y2, x1:x2] = inst_id
        instances_json[str(inst_id)] = int(b["class"])
        coords.append((inst_id, z1, z2, y1, y2, x1, x2))

    # Tamamen örtülen instance'lara merkez voksel zorla ata
    present = set(int(v) for v in np.unique(mask_arr) if v > 0)
    for inst_id, z1, z2, y1, y2, x1, x2 in coords:
        if inst_id not in present:
            zc = min((z1 + z2) // 2, mask_arr.shape[0] - 1)
            yc = min((y1 + y2) // 2, mask_arr.shape[1] - 1)
            xc = min((x1 + x2) // 2, mask_arr.shape[2] - 1)
            mask_arr[zc, yc, xc] = inst_id
            present.add(inst_id)

    mask_itk = sitk.GetImageFromArray(mask_arr)
    mask_itk.CopyInformation(img_itk)
    sitk.WriteImage(mask_itk, str(label_nii_path))
    label_json_path.write_text(json.dumps({"instances": instances_json}))
    return "ok"

# Geçersiz mask dosyalarını sil (nndet_prep'te hata veren vakalar)
_labels_dir = TASK_DIR / "labelsTr"
_broken = []
for _j in _labels_dir.glob("*.json"):
    _nii = _labels_dir / _j.name.replace(".json", ".nii.gz")
    if _nii.exists():
        _info = json.loads(_j.read_text())
        _inst_ids = set(int(k) for k in _info.get("instances", {}).keys())
        _mask_ids = set(int(v) for v in np.unique(sitk.GetArrayFromImage(sitk.ReadImage(str(_nii)))) if v > 0)
        if not _inst_ids.issubset(_mask_ids):
            _broken.append(_j.stem)
            _j.unlink(); _nii.unlink()
if _broken:
    print(f"⚠️  {len(_broken)} geçersiz vaka temizlendi: {_broken}")

results = [prep_case(c, "train") for c in tqdm(train_cases + val_cases, desc="prep train")]
ok   = results.count("ok")
skip = results.count("skip")
miss = [r for r in results if r.startswith("missing")]
print(f"✓ Yeni: {ok}, ⏭️  Atlandı: {skip}, ❌ Eksik NIfTI: {len(miss)}")
if miss: print("Eksik vakalar:", miss[:5])

In [ ]:
dataset_meta = {
    "task": TASK_NAME,
    "name": "TR_ABDOMEN_RAD_EMERGENCY",
    "target_class": None,
    "test_labels": False,
    "labels": {str(i): name for i, name in enumerate(SUPER_CLASSES)},
    "modalities": {"0": "CT"},
    "dim": 3,
}
(TASK_DIR.parent / "dataset.json").write_text(json.dumps(dataset_meta, indent=2))
print("dataset.json yazıldı:", TASK_DIR.parent / "dataset.json")
print(json.dumps(dataset_meta, indent=2, ensure_ascii=False))

## 5. nnDetection Plan + Preprocess

```bash
nndet_prep 100 --full_check
```

Patch boyutu, batch boyutu, foreground/background oranını otomatik belirler ve `splits_final.pkl` üretir.

In [ ]:
# nndet_prep için gerekli env değişkenleri
os.environ.setdefault("det_verbose",     "1")
os.environ.setdefault("det_num_threads", str(os.cpu_count() or 4))

# Mevcut yanlış formattaki JSON'ları temizle (.nii.gz çifti yoksa)
labels_dir = TASK_DIR / "labelsTr"
stale = [j for j in labels_dir.glob("*.json")
         if not (labels_dir / j.name.replace(".json", ".nii.gz")).exists()]
if stale:
    print(f"⚠️  {len(stale)} JSON'ın NIfTI maskesi yok → silindi, Cell 14'ü tekrar çalıştırın")
    for j in stale: j.unlink()

# nndet_prep binary'sini bul (nndet_venv veya PATH)
_nndet_venv = Path(__file__).parent / "nndet_venv" / "bin" / "nndet_prep" \
    if "__file__" in dir() else None
_nndet_bin = str(Path(sys.executable).parent / "nndet_prep")
nndet_prep_cmd = str(_nndet_venv) if (_nndet_venv and _nndet_venv.exists()) else _nndet_bin

print(f"nnDetection preprocess başlatılıyor... ({nndet_prep_cmd})")
try:
    r = subprocess.run(
        [nndet_prep_cmd, "100", "--full_check"],
        capture_output=True, text=True, timeout=3600*3, env=os.environ,
    )
    print("STDOUT:\n", r.stdout[-2000:])
    if r.returncode != 0:
        print("STDERR:\n", r.stderr[-2000:])
    else:
        print("✓ Preprocess tamamlandı!")
except FileNotFoundError:
    nndet_prep_abs = "/Users/ramazanpolat/Desktop/datasets/abdomen/nndet_venv/bin/nndet_prep"
    print(f"⚠️  nndet_prep bulunamadı. Terminalde çalıştırın:")
    print(f"  det_data='{os.environ['det_data']}' det_models='{os.environ['det_models']}' "
          f"det_verbose=1 det_num_threads={os.environ['det_num_threads']} "
          f"{nndet_prep_abs} 100 --full_check")
except subprocess.TimeoutExpired:
    print("Süre aşıldı; arka planda devam ediyor olabilir.")

## 6. Fold 0 Eğitim

In [ ]:
print("nnDetection train — fold 0, ~8-15 saat (T4)...")
try:
    r = subprocess.run(["nndet_train", "100", "--sweep", "-o", "exp.fold=0"],
                       capture_output=False, text=True, env=os.environ)
    print("Eğitim çıktı kodu:", r.returncode)
except FileNotFoundError:
    print("⚠️  Manuel: nndet_train 100 --sweep -o exp.fold=0")

## 7. 3D Çıkarım

In [ ]:
print("nnDetection predict — fold 0...")
try:
    r = subprocess.run(["nndet_predict", "100", "RetinaUNetV001_D3V001_3d",
                        "--fold", "0"],
                       capture_output=False, text=True, env=os.environ)
    print("Predict çıktı kodu:", r.returncode)
except FileNotFoundError:
    print("⚠️  Manuel çalıştırın.")

In [ ]:
from src.evaluation import top5_f1_mean, f1_at_iou
from PIL import Image
from src.config import DET_DATA_DIR

PRED_DIR = Path(os.environ["det_models"]) / TASK_NAME / "RetinaUNetV001_D3V001_3d" / "fold0" / "val_predictions"
print("Tahmin klasörü:", PRED_DIR, "→ var?", PRED_DIR.exists() if PRED_DIR.parent.exists() else "henüz yok")

def load_3d_preds_as_2d(pred_dir):
    rows = []
    if not pred_dir.exists(): return pd.DataFrame(rows)
    for pkl in pred_dir.glob("*_boxes.json"):
        case = int(pkl.stem.split("_")[1])
        data = json.loads(pkl.read_text())
        for b, s, c in zip(data.get("boxes",[]), data.get("scores",[]), data.get("classes",[])):
            x1,y1,z1,x2,y2,z2 = b
            for z in range(int(z1), int(z2)+1):
                rows.append({"case": case, "image_id": z, "class": int(c),
                             "score": float(s),
                             "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return pd.DataFrame(rows)

pred_df = load_3d_preds_as_2d(PRED_DIR)
print(f"3D→2D projeksiyon: {len(pred_df):,} satır")

In [ ]:
val_lbl_dir = DET_DATA_DIR / f"fold{fold}" / "labels" / "val"
val_img_dir = DET_DATA_DIR / f"fold{fold}" / "images" / "val"
gt_rows = []
for lp in val_lbl_dir.glob("*.txt"):
    ip = val_img_dir / (lp.stem + ".png")
    if not ip.exists(): continue
    W, H = Image.open(ip).size
    stem = lp.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5: continue
        cl = int(p[0]); cx,cy,w,h = map(float, p[1:5])
        gt_rows.append({"case": int(case), "image_id": int(image_id),
                        "class": cl,
                        "x1": (cx-w/2)*W, "y1": (cy-h/2)*H,
                        "x2": (cx+w/2)*W, "y2": (cy+h/2)*H})
gt_df = pd.DataFrame(gt_rows)
print(f"GT: {len(gt_df):,}")

if not pred_df.empty:
    result = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)
    print(f"\nnnDetection fold 0 — Top-5 F1: {result['top5_mean_f1']:.4f}")
    print(f"Macro F1 @ IoU 0.3: {detail['macro_f1']:.4f}")
else:
    print("Tahmin yok — predict adımının bitmesini bekleyin.")

## 8. Faz 4 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| NIfTI hacimler | `outputs/nndet/nifti/case_*.nii.gz` |
| Task verisi | `outputs/nndet/raw/Task100_Abdomen/raw_splitted/` |
| Eğitilmiş model | `outputs/nndet/models/Task100_Abdomen/RetinaUNetV001_D3V001_3d/fold0/` |
| 3D tahminler | `.../fold0/val_predictions/*_boxes.json` |

**Sonraki:** `Faz5_Harici_Test_Yarisma.ipynb` (YOLO 2D vs nnDetection 3D karşılaştırma).